# 2.1 Embeddings Textuels

Encodage des descriptions de produits avec deux méthodes :
- **Méthode 1** : TF-IDF (baseline)
- **Méthode 2** : Embeddings neuronaux multilingues

Calcul des matrices de similarité cosinus pour chaque méthode.

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# Charger les données
df = pd.read_csv('../data/subset_fashion_dataset/products_final.csv')
print(f"Dataset chargé : {len(df)} produits")
df.head()


Dataset chargé : 400 produits


,id,nom,categorie,sous_categorie,couleur,image_path,description,prix,matiere
0,15832,Visudh Pink Printed Ethnic Kurta,Topwear,Kurtas,Pink,images/15832.jpg,Le Visudh Pink Printed Ethnic Kurta est un hau...,19228,Coton bio
1,33968,Femella Women Gold Shirt,Topwear,Shirts,Gold,images/33968.jpg,"La chemise Femella pour femmes, d'un éclat dor...",18138,Coton
2,3314,Guerrilla Men's Warrior Brown T-shirt,Topwear,Tshirts,Brown,images/3314.jpg,Le t-shirt Warrior Guerrilla pour homme arbore...,14370,Mélange coton
3,11767,Lee Men Printed Grey Tshirts,Topwear,Tshirts,Grey,images/11767.jpg,Lee Men Printed Grey Tshirts : T-shirt en gris...,19269,Polyester
4,8725,Indigo Nation Men Checks Shirt Black Shirts,Topwear,Shirts,Black,images/8725.jpg,L’Indigo Nation Men Checks Black Shirt allie é...,12233,Mélange coton


In [2]:
# Préparer le texte : nom + description
df['text'] = df['nom'] + ' ' + df['description'].fillna('')

# Afficher un exemple
print("Exemple de texte combiné :")
print(df['text'].iloc[0])

Exemple de texte combiné :
Visudh Pink Printed Ethnic Kurta Le Visudh Pink Printed Ethnic Kurta est un haut en coton orné d’un imprimé ethnique délicat, pour un look à la fois traditionnel et moderne. Sa coupe ample et son rose vif en font un choix idéal pour une tenue confortable et esthétique, parfaite pour les occasions décontractées ou festives.


## Méthode 1 : TF-IDF (Baseline)

In [3]:
# TF-IDF Vectorizer
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words=None,
    lowercase=True
)

# Créer la matrice TF-IDF
tfidf_matrix = tfidf.fit_transform(df['text'])

print(f"Matrice TF-IDF : {tfidf_matrix.shape}")
print(f"Type : {type(tfidf_matrix)} (sparse matrix)")

Matrice TF-IDF : (400, 1664)
Type : <class 'scipy.sparse._csr.csr_matrix'> (sparse matrix)


In [4]:
# Matrice de similarité cosinus (TF-IDF)
tfidf_similarity = cosine_similarity(tfidf_matrix)

print(f"Matrice de similarité TF-IDF : {tfidf_similarity.shape}")
print("\nExemple de similarités pour le produit 0 (top 5) :")
sim_scores = tfidf_similarity[0]
top_indices = np.argsort(sim_scores)[::-1][1:6]  # Top 5 (exclure soi-même)
for idx in top_indices:
    print(f"  - Produit {idx}: {sim_scores[idx]:.3f} - {df['nom'].iloc[idx]}")

Matrice de similarité TF-IDF : (400, 400)

Exemple de similarités pour le produit 0 (top 5) :
  - Produit 36: 0.496 - W Women Printed Pink Kurta
  - Produit 29: 0.365 - Vishudh Women Pink Pigment Printed Kurta
  - Produit 35: 0.271 - Mother Earth Women Black Kurta
  - Produit 30: 0.267 - Mother Earth Women Printed Blue Kurta
  - Produit 44: 0.249 - Vishudh Women Red & White Kurta


## Méthode 2 : Embeddings Neuronaux (Sentence Transformers)

In [5]:
# Charger le modèle multilingue
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
print(f"Modèle chargé : paraphrase-multilingual-MiniLM-L12-v2")

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 40c7ffcf-75b3-4555-8f7f-cd26cc9814d6)')' thrown while requesting HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/e8f8c211226b894fcb81acc59f3b34ba3efd5f42/modules.json
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: e199a58b-9439-4a3e-9551-dc1daf7de40b)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/config_sentence_transformers.json
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: a8e8abb6-aaa7-40ac-8a45-fa64cb693bf8)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingua

Modèle chargé : paraphrase-multilingual-MiniLM-L12-v2


In [6]:
# Encoder les textes
print("Encodage en cours...")
embeddings = model.encode(df['text'].tolist(), show_progress_bar=True)

print(f"\nMatrice d'embeddings : {embeddings.shape}")
print(f"Dimension des vecteurs : {embeddings.shape[1]}")

Encodage en cours...


Batches:   0%|          | 0/13 [00:00<?, ?it/s]


Matrice d'embeddings : (400, 384)
Dimension des vecteurs : 384


In [7]:
# Matrice de similarité cosinus (Embeddings)
embedding_similarity = cosine_similarity(embeddings)

print(f"Matrice de similarité Embeddings : {embedding_similarity.shape}")
print("\nExemple de similarités pour le produit 0 (top 5) :")
sim_scores_emb = embedding_similarity[0]
top_indices_emb = np.argsort(sim_scores_emb)[::-1][1:6]
for idx in top_indices_emb:
    print(f"  - Produit {idx}: {sim_scores_emb[idx]:.3f} - {df['nom'].iloc[idx]}")

Matrice de similarité Embeddings : (400, 400)

Exemple de similarités pour le produit 0 (top 5) :
  - Produit 29: 0.868 - Vishudh Women Pink Pigment Printed Kurta
  - Produit 36: 0.843 - W Women Printed Pink Kurta
  - Produit 44: 0.740 - Vishudh Women Red & White Kurta
  - Produit 21: 0.693 - Fabindia Women White Kurta
  - Produit 30: 0.686 - Mother Earth Women Printed Blue Kurta


## Comparaison des deux méthodes

In [8]:
# Comparaison pour un produit exemple
product_idx = 0

print(f"=== Produit : {df['nom'].iloc[product_idx]} ===")
print(f"Catégorie : {df['categorie'].iloc[product_idx]}")
print()

# Top 5 TF-IDF
print("TOP 5 TF-IDF :")
top_tfidf = np.argsort(tfidf_similarity[product_idx])[::-1][1:6]
for i, idx in enumerate(top_tfidf, 1):
    print(f"{i}. [{df['categorie'].iloc[idx]}] {df['nom'].iloc[idx]} ({tfidf_similarity[product_idx][idx]:.3f})")

print()

# Top 5 Embeddings
print("TOP 5 EMBEDDINGS :")
top_emb = np.argsort(embedding_similarity[product_idx])[::-1][1:6]
for i, idx in enumerate(top_emb, 1):
    print(f"{i}. [{df['categorie'].iloc[idx]}] {df['nom'].iloc[idx]} ({embedding_similarity[product_idx][idx]:.3f})")

=== Produit : Visudh Pink Printed Ethnic Kurta ===
Catégorie : Topwear

TOP 5 TF-IDF :
1. [Topwear] W Women Printed Pink Kurta (0.496)
2. [Topwear] Vishudh Women Pink Pigment Printed Kurta (0.365)
3. [Topwear] Mother Earth Women Black Kurta (0.271)
4. [Topwear] Mother Earth Women Printed Blue Kurta (0.267)
5. [Topwear] Vishudh Women Red & White Kurta (0.249)

TOP 5 EMBEDDINGS :
1. [Topwear] Vishudh Women Pink Pigment Printed Kurta (0.868)
2. [Topwear] W Women Printed Pink Kurta (0.843)
3. [Topwear] Vishudh Women Red & White Kurta (0.740)
4. [Topwear] Fabindia Women White Kurta (0.693)
5. [Topwear] Mother Earth Women Printed Blue Kurta (0.686)


## Sauvegarde des résultats

In [9]:
# Sauvegarder les matrices de similarité
np.save('../results/tfidf_similarity_matrix.npy', tfidf_similarity)
np.save('../results/embedding_similarity_matrix.npy', embedding_similarity)

# Sauvegarder les embeddings
np.save('../results/text_embeddings.npy', embeddings)

print("Matrices sauvegardées dans results/ :")
print("  - tfidf_similarity_matrix.npy")
print("  - embedding_similarity_matrix.npy")
print("  - text_embeddings.npy")

Matrices sauvegardées dans results/ :
  - tfidf_similarity_matrix.npy
  - embedding_similarity_matrix.npy
  - text_embeddings.npy


## Extraction des Top-5 pour les Ancres (Ground Truth)

In [10]:
# Charger le ground truth
ground_truth = pd.read_csv('../data/subset_fashion_dataset/ground_truth.csv', sep=';')
print(f"Ground truth chargé : {len(ground_truth)} ancres")
ground_truth.head()

Ground truth chargé : 30 ancres


,anchor_id,global_id1,global_id2,global_id3,global_id4,global_id5,visual_id1,visual_id2,visual_id3,visual_id4,visual_id5,semantic_id1,semantic_id2,semantic_id3,semantic_id4,semantic_id5
0,3314,4861,26486,24146,31279,38808,4861,26486,31279,24146,38808,24146,37148,11767,38808,8356
1,4543,4445,15726,6852,5739,26614,4445,5739,15743,6852,34832,6852,4445,15726,4487,5739
2,15270,14498,49483,6431,3795,6495,6431,3795,14498,17041,36476,6431,14498,49483,14500,41832
3,17993,27906,7885,12677,48391,51365,12677,7885,48497,27906,26822,2113,7885,11279,41993,22786
4,20610,53921,43227,36069,38315,35915,36069,43227,53921,35915,38315,53886,38315,53921,14799,24736


In [11]:
# Mapping : ID produit → index dans la matrice de similarité
id_to_index = {row['id']: idx for idx, row in df.iterrows()}
print(f"Mapping créé : {len(id_to_index)} produits")
print(f"Exemple : ID 3314 → index {id_to_index[3314]}")

Mapping créé : 400 produits
Exemple : ID 3314 → index 2


In [12]:
# Fonction pour extraire top-5 voisins
def get_top5_neighbors(anchor_id, similarity_matrix, id_to_index, df):
    """Récupère les top 5 voisins pour un produit ancre"""
    anchor_idx = id_to_index[anchor_id]
    sim_scores = similarity_matrix[anchor_idx]
    
    # Trier et prendre top 5 (exclure soi-même)
    top_indices = np.argsort(sim_scores)[::-1][1:6]
    
    # Retourner les IDs des produits
    top_ids = [df.iloc[idx]['id'] for idx in top_indices]
    top_scores = [sim_scores[idx] for idx in top_indices]
    
    return list(zip(top_ids, top_scores))

# Test sur une ancre
test_anchor = ground_truth['anchor_id'].iloc[0]
print(f"Test sur l'ancre {test_anchor} :")
print("\nTF-IDF :")
top5_tfidf = get_top5_neighbors(test_anchor, tfidf_similarity, id_to_index, df)
for prod_id, score in top5_tfidf:
    print(f"  {prod_id} ({score:.3f})")

print("\nEmbeddings :")
top5_emb = get_top5_neighbors(test_anchor, embedding_similarity, id_to_index, df)
for prod_id, score in top5_emb:
    print(f"  {prod_id} ({score:.3f})")

Test sur l'ancre 3314 :

TF-IDF :
  41403 (0.226)
  42394 (0.205)
  38808 (0.201)
  39345 (0.199)
  26486 (0.193)

Embeddings :
  26486 (0.694)
  31279 (0.647)
  8834 (0.647)
  26821 (0.638)
  9636 (0.628)


In [13]:
# Extraire top-5 pour toutes les ancres
results_tfidf = []
results_embeddings = []

for anchor_id in ground_truth['anchor_id']:
    # TF-IDF
    top5_tfidf = get_top5_neighbors(anchor_id, tfidf_similarity, id_to_index, df)
    results_tfidf.append({
        'anchor_id': anchor_id,
        'top5_ids': [x[0] for x in top5_tfidf],
        'top5_scores': [x[1] for x in top5_tfidf]
    })
    
    # Embeddings
    top5_emb = get_top5_neighbors(anchor_id, embedding_similarity, id_to_index, df)
    results_embeddings.append({
        'anchor_id': anchor_id,
        'top5_ids': [x[0] for x in top5_emb],
        'top5_scores': [x[1] for x in top5_emb]
    })

# Créer DataFrames
df_tfidf = pd.DataFrame(results_tfidf)
df_embeddings = pd.DataFrame(results_embeddings)

print(f"Résultats TF-IDF : {df_tfidf.shape}")
print(f"Résultats Embeddings : {df_embeddings.shape}")
print("\nExemple TF-IDF :")
print(df_tfidf.head(2))

Résultats TF-IDF : (30, 3)
Résultats Embeddings : (30, 3)

Exemple TF-IDF :
   anchor_id                             top5_ids  \
0       3314  [41403, 42394, 38808, 39345, 26486]   
1       4543   [15726, 59338, 15743, 19308, 4445]   

                                         top5_scores  
0  [0.22613143667517965, 0.20472415654436693, 0.2...  
1  [0.3870939188164906, 0.2097120741772003, 0.203...  


In [14]:
# Sauvegarder les résultats
df_tfidf.to_csv('../results/tfidf_top5_anchors.csv', index=False)
df_embeddings.to_csv('../results/embeddings_top5_anchors.csv', index=False)

print("Résultats sauvegardés :")
print("  - results/tfidf_top5_anchors.csv")
print("  - results/embeddings_top5_anchors.csv")

# Afficher un exemple de comparaison
print("\n" + "="*60)
print(f"Comparaison pour l'ancre {test_anchor} :")
print("="*60)

# Ground truth
gt_row = ground_truth[ground_truth['anchor_id'] == test_anchor].iloc[0]
print("\nGround Truth (global) :")
for i in range(1, 6):
    print(f"  {i}. {gt_row[f'global_id{i}']}")

# TF-IDF
print("\nTF-IDF :")
for i, (prod_id, score) in enumerate(top5_tfidf, 1):
    print(f"  {i}. {prod_id} ({score:.3f})")

# Embeddings
print("\nEmbeddings :")
for i, (prod_id, score) in enumerate(top5_emb, 1):
    print(f"  {i}. {prod_id} ({score:.3f})")

Résultats sauvegardés :
  - results/tfidf_top5_anchors.csv
  - results/embeddings_top5_anchors.csv

Comparaison pour l'ancre 3314 :

Ground Truth (global) :
  1. 4861
  2. 26486
  3. 24146
  4. 31279
  5. 38808

TF-IDF :
  1. 47630 (0.510)
  2. 57007 (0.333)
  3. 10203 (0.241)
  4. 24077 (0.235)
  5. 17319 (0.217)

Embeddings :
  1. 47630 (0.743)
  2. 57007 (0.731)
  3. 24077 (0.709)
  4. 56891 (0.697)
  5. 45785 (0.669)
